# Clarabel vs PuLPThis notebook compares [Clarabel](https://github.com/oxfordcontrol/Clarabel.rs) — a Rust-basedinterior-point solver for conic optimization — against PuLP (with CBC) on deterministic LPs and QPs.It checks objective agreement, primal feasibility, and solver timing.The problem suite covers:- Pure equality-constrained LPs- Mixed equality/inequality LPs- Constrained QPs (positive-definite quadratic objective)

## Prerequisites```bashpip install clarabel pulp scipy```

## Setup

In [ ]:
from pathlib import Pathimport importlib.metadataimport timeimport numpy as npROOT = Path.cwd()while not (ROOT / 'CMakeLists.txt').is_file():    ROOT = ROOT.parentprint(f'repository : {ROOT}')print(f'Clarabel   : {importlib.metadata.version("clarabel")}')print(f'PuLP       : {importlib.metadata.version("pulp")}')

## Problem generationProblems are feasible by construction: a point strictly inside finite boundsis sampled first, then the right-hand side is derived from it.

In [ ]:
import scipy.sparse as spimport clarabeldef make_equality_lp(seed: int, n: int = 12, m: int = 5) -> dict:    """Feasible LP in equality form with finite bounds."""    rng = np.random.default_rng(seed)    lb = rng.uniform(-1.0, 0.0, size=n)    ub = lb + rng.uniform(2.0, 6.0, size=n)    x_feas = lb + rng.uniform(0.2, 0.8, size=n) * (ub - lb)    A = rng.normal(size=(m, n))    return {        "name": f"eq_lp_{seed}",        "A": np.ascontiguousarray(A, dtype=np.float64),        "b": A @ x_feas,        "c": rng.normal(size=n),        "lb": lb.copy(),        "ub": ub.copy(),        "type": "lp",        "senses": ["="] * m,    }def make_mixed_lp(seed: int, n: int = 12, m: int = 6) -> dict:    """LP with mixed equality and inequality constraints."""    rng = np.random.default_rng(seed)    lb = rng.uniform(-1.0, 0.0, size=n)    ub = lb + rng.uniform(2.0, 6.0, size=n)    x_feas = lb + 0.5 * (ub - lb)    A = rng.normal(size=(m, n))    senses = np.array(["=", "<=", ">=", "<=", ">=", "="], dtype=object)    ax = A @ x_feas    slack = rng.uniform(0.2, 1.0, size=m)    b = np.where(        senses == "<=",        ax + slack,        np.where(senses == ">=", ax - slack, ax),    )    return {        "name": f"mixed_lp_{seed}",        "A": np.ascontiguousarray(A, dtype=np.float64),        "b": b.astype(np.float64),        "c": rng.normal(size=n),        "lb": lb.copy(),        "ub": ub.copy(),        "type": "lp",        "senses": senses.tolist(),    }def make_qp(seed: int, n: int = 8, m: int = 4) -> dict:    """QP with positive-definite quadratic objective and linear constraints."""    rng = np.random.default_rng(seed)    lb = rng.uniform(-2.0, 0.0, size=n)    ub = lb + rng.uniform(3.0, 8.0, size=n)    x_feas = lb + rng.uniform(0.2, 0.8, size=n) * (ub - lb)    # Positive-definite P: Q^T D Q with D > 0    Q = rng.normal(size=(n, n))    _, qr = np.linalg.qr(Q)    D = np.diag(rng.uniform(1.0, 5.0, size=n))    P = qr.T @ D @ qr  # symmetric positive definite    q = rng.normal(size=n)    A = rng.normal(size=(m, n))    b = A @ x_feas    return {        "name": f"qp_{seed}",        "P": np.ascontiguousarray(P, dtype=np.float64),        "q": q.copy(),        "A": np.ascontiguousarray(A, dtype=np.float64),        "b": b.copy(),        "lb": lb.copy(),        "ub": ub.copy(),        "type": "qp",        "senses": ["="] * m,    }# Build test suitelp_problems = [make_equality_lp(s) for s in range(5)]mixed_problems = [make_mixed_lp(s) for s in range(3)]qp_problems = [make_qp(s) for s in range(3)]print(f"Test suite: {len(lp_problems)} equality LPs, "      f"{len(mixed_problems)} mixed LPs, {len(qp_problems)} QPs")

## Solver adapters### Clarabel adapterClarabel solves the conic form:```min  0.5 x^T P x + q^T xs.t. l <= A x <= u   (via conic decomposition)```**Important:** Clarabel has no native variable bounds. We encode`lb <= x <= ub` as two additional rows per variable:- `-x_j <= -lb_j` (i.e., `x_j >= lb_j`)- `x_j <= ub_j`Equality constraints use `ZeroConeT(n)`, inequalities use `NonnegativeConeT(n)`.

In [ ]:
def solve_clarabel(problem: dict, tol: float = 1e-8) -> dict:
    """Solve with Clarabel interior-point solver."""
    if problem['type'] == 'lp':
        P = sp.csc_matrix((problem['A'].shape[1], problem['A'].shape[1]))
        q = problem['c'].copy()
    else:
        P = sp.csc_matrix(problem['P'])
        q = problem['q'].copy()

    A = sp.csc_matrix(problem['A'])
    b = problem['b'].copy()

    # Build conic decomposition: separate equality and inequality rows
    # Use index-based extraction because mixed-sense constraints are
    # interleaved — contiguous slicing would misassign rows.
    A_rows = []
    b_rows = []
    cones = []

    # Equality rows
    eq_idx = [i for i, s in enumerate(problem['senses']) if s == '=']
    if eq_idx:
        A_rows.append(A[eq_idx, :].copy())
        b_rows.append(b[eq_idx].copy())
        cones.append(clarabel.ZeroConeT(len(eq_idx)))

    # <= rows
    le_idx = [i for i, s in enumerate(problem['senses']) if s == '<=']
    if le_idx:
        A_rows.append(A[le_idx, :].copy())
        b_rows.append(b[le_idx].copy())
        cones.append(clarabel.NonnegativeConeT(len(le_idx)))

    # >= rows (convert to <=: negate both sides)
    ge_idx = [i for i, s in enumerate(problem['senses']) if s == '>=']
    if ge_idx:
        A_rows.append(-A[ge_idx, :].copy())
        b_rows.append(-b[ge_idx].copy())
        cones.append(clarabel.NonnegativeConeT(len(ge_idx)))

    n = problem['A'].shape[1]
    lb = problem['lb']
    ub = problem['ub']

    # Add lower bounds: -x_j <= -lb_j  (i.e., x_j >= lb_j)
    if lb is not None:
        A_rows.append(-sp.eye(n, format='csc'))
        b_rows.append(-lb)
        cones.append(clarabel.NonnegativeConeT(n))

    # Add upper bounds: x_j <= ub_j
    if ub is not None:
        A_rows.append(sp.eye(n, format='csc'))
        b_rows.append(ub)
        cones.append(clarabel.NonnegativeConeT(n))

    A_combined = sp.vstack(A_rows, format='csc')
    b_combined = np.concatenate(b_rows)

    settings = clarabel.DefaultSettings()
    settings.max_iter = 200
    settings.verbose = False
    settings.tol_feas = tol
    settings.tol_gap_rel = tol

    solver = clarabel.DefaultSolver(
        P, q, A_combined, b_combined, cones, settings
    )
    started = time.perf_counter()
    result = solver.solve()
    elapsed_ms = 1e3 * (time.perf_counter() - started)

    # result.status is a SolverStatus enum; compare with enum values
    SS = clarabel.SolverStatus
    if result.status == SS.Solved:
        status = 'optimal'
    elif result.status == SS.NearlySolved:
        status = 'near_optimal'
    elif result.status == SS.PrimalInfeasible:
        status = 'infeasible'
    elif result.status == SS.DualInfeasible:
        status = 'unbounded'
    elif result.status == SS.MaxIterations:
        status = 'iter_limit'
    elif result.status == SS.MaxTime:
        status = 'time_limit'
    else:
        status = str(result.status).lower()
    return {
        'solver': 'Clarabel',
        'status': status,
        'objective': (float(result.obj_val)
                      if result.obj_val is not None else np.nan),
        'x': result.x.copy() if result.x is not None else None,
        'time_ms': elapsed_ms,
        'iterations': result.iterations,
    }

### PuLP adapterPuLP with CBC backend — an LP-only MIP solver.

In [ ]:
import pulpdef solve_pulp(problem: dict) -> dict:    """Solve with PuLP CBC backend."""    cbc = pulp.PULP_CBC_CMD(msg=False, timeLimit=30)    if not cbc.available():        return {"solver": "PuLP/CBC", "status": "unavailable",                "objective": np.nan, "x": None, "time_ms": 0}    model = pulp.LpProblem(problem["name"], pulp.LpMinimize)    xs = [        pulp.LpVariable(f"x_{j}", lowBound=float(lo), upBound=float(hi))        for j, (lo, hi) in enumerate(zip(problem["lb"], problem["ub"]))    ]    if problem["type"] == "lp":        model += pulp.lpSum(c * x for c, x in zip(problem["c"], xs))    for i, (row, rhs, sense) in enumerate(        zip(problem["A"], problem["b"], problem["senses"])    ):        expr = pulp.lpSum(c * x for c, x in zip(row, xs))        if sense == "=":            model += expr == float(rhs), f"eq_{i}"        elif sense == "<=":            model += expr <= float(rhs), f"le_{i}"        elif sense == ">=":            model += expr >= float(rhs), f"ge_{i}"    started = time.perf_counter()    status_code = model.solve(cbc)    elapsed_ms = 1e3 * (time.perf_counter() - started)    status = pulp.LpStatus[status_code]    if status != "Optimal":        return {"solver": "PuLP/CBC", "status": status,                "objective": np.nan, "x": None, "time_ms": elapsed_ms}    x = np.asarray([pulp.value(v) for v in xs], dtype=np.float64)    obj = problem["c"] @ x    return {        "solver": "PuLP/CBC",        "status": status,        "objective": float(obj),        "x": x,        "time_ms": elapsed_ms,    }

## Feasibility checkerIndependent residual computation — never reads solver status.

In [ ]:
def max_primal_violation(problem: dict, x: np.ndarray) -> float:    """Maximum constraint and bound violation for x."""    if x is None or not np.all(np.isfinite(x)):        return np.inf    lhs = problem["A"] @ x    violations = [        np.max(np.maximum(problem["lb"] - x, 0.0)),        np.max(np.maximum(x - problem["ub"], 0.0)),    ]    for value, rhs, sense in zip(lhs, problem["b"], problem["senses"]):        if sense == "=":            violations.append(abs(value - rhs))        elif sense == "<=":            violations.append(max(value - rhs, 0.0))        else:            violations.append(max(rhs - value, 0.0))    return float(max(violations))OBJECTIVE_RTOL = 1e-5FEASIBILITY_ATOL = 1e-5def compare(problem: dict) -> dict:    clarabel_res = solve_clarabel(problem)    pulp_res = solve_pulp(problem)    scale = (1.0 + abs(pulp_res["objective"])             if pulp_res["objective"] is not None else 1.0)    pulp_x = pulp_res["x"]    clarabel_x = clarabel_res["x"]    x_dist = (float(np.linalg.norm(clarabel_x - pulp_x, ord=np.inf))              if clarabel_x is not None and pulp_x is not None else np.inf)    return {        "problem": problem["name"],        "type": problem["type"],        "clarabel_status": clarabel_res["status"],        "pulp_status": pulp_res["status"],        "clarabel_obj": clarabel_res["objective"],        "pulp_obj": pulp_res["objective"],        "rel_obj_gap": abs(clarabel_res["objective"] - pulp_res["objective"])                       / scale,        "clarabel_viol": max_primal_violation(problem, clarabel_x),        "pulp_viol": max_primal_violation(problem, pulp_x),        "clarabel_ms": clarabel_res["time_ms"],        "pulp_ms": pulp_res["time_ms"],        "clarabel_iters": clarabel_res.get("iterations", 0),        "x_inf_dist": x_dist,    }def print_lp_table(rows: list) -> None:    header = (        f"{'problem':<18} {'type':>4} "        f"{'Clar obj':>12} {'PuLP obj':>12} {'rel gap':>10} "        f"{'Clar viol':>11} {'Clar ms':>8} {'PuLP ms':>8}"    )    print(header)    print("-" * len(header))    for r in rows:        print(            f"{r['problem']:<18} {r['type']:>4} "            f"{r['clarabel_obj']:>12.6g} {r['pulp_obj']:>12.6g} "            f"{r['rel_obj_gap']:>10.3e} {r['clarabel_viol']:>11.3e} "            f"{r['clarabel_ms']:>8.2f} {r['pulp_ms']:>8.2f}"        )

## Results### Equality-constrained LPs

In [ ]:
lp_results = [compare(p) for p in lp_problems]print_lp_table(lp_results)for r in lp_results:    assert r["pulp_status"] == "Optimal", r    assert r["clarabel_status"] in ("optimal", "near_optimal"), r    assert np.isfinite(r["clarabel_obj"]), r    assert r["rel_obj_gap"] <= OBJECTIVE_RTOL, r    assert r["clarabel_viol"] <= FEASIBILITY_ATOL, rn = len(lp_results)print(f"PASS: {n} equality LPs -- Clarabel matches PuLP/CBC")

### Mixed-inequality LPs

In [ ]:
mixed_results = [compare(p) for p in mixed_problems]print_lp_table(mixed_results)for r in mixed_results:    assert r["pulp_status"] == "Optimal", r    assert r["clarabel_status"] in ("optimal", "near_optimal"), r    assert np.isfinite(r["clarabel_obj"]), r    assert r["rel_obj_gap"] <= OBJECTIVE_RTOL, r    assert r["clarabel_viol"] <= FEASIBILITY_ATOL, rn = len(mixed_results)print(f"PASS: {n} mixed LPs -- Clarabel matches PuLP/CBC")

### QPsPuLP/CBC is an LP-only solver and cannot handle quadratic objectives.We compare Clarabel against scipy's SLSQP instead.

In [ ]:
from scipy.optimize import minimizedef solve_scipy_qp(problem: dict) -> dict:    """Solve QP with scipy SLSQP as reference."""    lb = problem["lb"]    ub = problem["ub"]    senses = problem["senses"]    A, b = problem["A"], problem["b"]    def objective(x):        return 0.5 * x @ problem["P"] @ x + problem["q"] @ x    def grad(x):        return problem["P"] @ x + problem["q"]    # Build constraints with explicit rhs values to avoid closure bugs    # (rhs would otherwise capture the last loop iteration value).    cons = []    for i, sense in enumerate(senses):        if sense == "=":            rhs_val = float(b[i])            cons.append({"type": "eq",                         "fun": lambda x, _i=i, _r=rhs_val: A[_i] @ x - _r})        elif sense == "<=":            rhs_val = float(b[i])            cons.append({"type": "ineq",                         "fun": lambda x, _i=i, _r=rhs_val: _r - A[_i] @ x})        elif sense == ">=":            rhs_val = float(b[i])            cons.append({"type": "ineq",                         "fun": lambda x, _i=i, _r=rhs_val: A[_i] @ x - _r})    x0 = (lb + ub) / 2    started = time.perf_counter()    result = minimize(        objective, x0, jac=grad, method="SLSQP",        bounds=list(zip(lb, ub)), constraints=cons,        options={"maxiter": 500, "ftol": 1e-10},    )    elapsed_ms = 1e3 * (time.perf_counter() - started)    if not result.success or result.fun is None:        return {"solver": "scipy/SLSQP", "status": "failed",                "objective": np.nan, "x": None, "time_ms": elapsed_ms}    return {        "solver": "scipy/SLSQP",        "status": "optimal" if result.success else "failed",        "objective": float(result.fun),        "x": result.x.copy(),        "time_ms": elapsed_ms,    }def compare_qp(problem: dict) -> dict:    clarabel_res = solve_clarabel(problem)    scipy_res = solve_scipy_qp(problem)    scale = (1.0 + abs(scipy_res["objective"])             if scipy_res["objective"] is not None else 1.0)    clarabel_x = clarabel_res["x"]    scipy_x = scipy_res["x"]    x_dist = (float(np.linalg.norm(clarabel_x - scipy_x, ord=np.inf))              if clarabel_x is not None and scipy_x is not None else np.inf)    return {        "problem": problem["name"],        "clarabel_status": clarabel_res["status"],        "scipy_status": scipy_res["status"],        "clarabel_obj": clarabel_res["objective"],        "scipy_obj": scipy_res["objective"],        "rel_obj_gap": abs(clarabel_res["objective"] - scipy_res["objective"])                       / scale,        "clarabel_viol": max_primal_violation(problem, clarabel_x),        "scipy_viol": max_primal_violation(problem, scipy_x),        "clarabel_ms": clarabel_res["time_ms"],        "scipy_ms": scipy_res["time_ms"],        "x_inf_dist": x_dist,    }def print_qp_table(rows: list) -> None:    _w = {"problem": 18, "Clar obj": 12, "SciPy obj": 12,          "rel gap": 10, "Clar viol": 11, "Clar ms": 8, "SciPy ms": 8}    cols = ["problem", "Clar obj", "SciPy obj", "rel gap", "Clar viol", "Clar ms", "SciPy ms"]    header = " ".join(f"{c:>{_w[c]}}" for c in cols)    print(header)    print("-" * len(header))    for r in rows:        print(            f"{r['problem']:<18} "            f"{r['clarabel_obj']:>12.6g} {r['scipy_obj']:>12.6g} "            f"{r['rel_obj_gap']:>10.3e} {r['clarabel_viol']:>11.3e} "            f"{r['clarabel_ms']:>8.2f} {r['scipy_ms']:>8.2f}"        )

In [ ]:
qp_results = [compare_qp(p) for p in qp_problems]print_qp_table(qp_results)for r in qp_results:    assert r["scipy_status"] == "optimal", r    assert r["clarabel_status"] in ("optimal", "near_optimal"), r    assert np.isfinite(r["clarabel_obj"]), r    assert r["rel_obj_gap"] <= OBJECTIVE_RTOL, r    assert r["clarabel_viol"] <= FEASIBILITY_ATOL, rn = len(qp_results)print(f"PASS: {n} QPs -- Clarabel matches scipy/SLSQP")

## Summary table

In [ ]:
all_rows = []for r in lp_results:    all_rows.append({**r, "ref_solver": "PuLP/CBC"})for r in qp_results:    all_rows.append({**r, "ref_solver": "scipy/SLSQP"})print(f"{'problem':<18} {'Clar obj':>12} {'Ref obj':>12} "      f"{'rel gap':>10} {'Clar viol':>11} {'Ref':>14}")print("-" * 78)for r in all_rows:    obj_ref = r.get("pulp_obj", r.get("scipy_obj", np.nan))    viol_ref = r.get("pulp_viol", r.get("scipy_viol", np.inf))    ref_name = "PuLP/CBC" if "pulp_obj" in r else "scipy/SLSQP"    print(        f"{r['problem']:<18} "        f"{r['clarabel_obj']:>12.6g} {obj_ref:>12.6g} "        f"{r['rel_obj_gap']:>10.3e} {r['clarabel_viol']:>11.3e} "        f"{ref_name:>14}"    )total = len(all_rows)passed = sum(    1 for r in all_rows    if r["rel_obj_gap"] <= OBJECTIVE_RTOL    and r["clarabel_viol"] <= FEASIBILITY_ATOL)print(f"Total: {passed}/{total} cases pass "      f"(obj gap < {OBJECTIVE_RTOL}, viol < {FEASIBILITY_ATOL})")